<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/email-management-api/blob/main/qwen25_email_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
!pip install -q pandas openpyxl
!pip install -U bitsandbytes>=0.46.1
!pip install -q -U trl

In [2]:
import pandas as pd
import json
import ast
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

Get Excel Sheet

In [3]:
from google.colab import files
uploaded = files.upload()
excel_file = next(iter(uploaded.keys()))
print("Uploaded:", excel_file)

Saving manual_classification_updated.xlsx to manual_classification_updated.xlsx
Uploaded: manual_classification_updated.xlsx


Extract the data from sheet

In [4]:
df = pd.read_excel(excel_file)

required_columns = [
    "subject",
    "from",
    "body",
    "sender_type",
    "is_action_required",
    "labels_to_remove",
    "labels_to_add"
]

df = df[required_columns]

print("Total records:", len(df))

Total records: 1981


In [5]:
def parse_labels(value):

    if isinstance(value, list):
        return value

    if pd.isna(value) or str(value).strip() == "":
        return []

    return ast.literal_eval(value)


def convert_record(row):

    input_data = {
        "subject": row["subject"],
        "from": row["from"],
        "body": row["body"],
        "sender_type": row["sender_type"],
        "is_action_required": row["is_action_required"]
    }

    output_data = {
        "labels_to_remove": parse_labels(row["labels_to_remove"]),
        "labels_to_add": parse_labels(row["labels_to_add"])
    }

    return {
        "input": json.dumps(input_data, ensure_ascii=False),
        "output": json.dumps(output_data, ensure_ascii=False)
    }


dataset = [
    convert_record(row)
    for _, row in df.iterrows()
]

print("Conversion completed!")
print("Total records:", len(dataset))

Conversion completed!
Total records: 1981


Save records as json

In [6]:
with open(
    "converted_dataset.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for record in dataset:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("JSONL file created successfully!")

JSONL file created successfully!


System Prompt

In [7]:

all_labels = set()

for column in ["labels_to_add", "labels_to_remove"]:
    for value in df[column]:
        all_labels.update(parse_labels(value))

ALLOWED_LABELS = sorted(all_labels)

print("Total unique labels:", len(ALLOWED_LABELS))

Total unique labels: 47


In [8]:
SYSTEM_PROMPT = f"""
You are an AI email management assistant.

Analyze the provided email and determine:

1. Which labels should be added.
2. Which existing labels should be removed.
3. Whether the email requires an action or reply from the user.

Allowed labels:
{json.dumps(ALLOWED_LABELS, ensure_ascii=False)}

Rules:
- Use only labels from the allowed labels list.
- An email can have multiple labels.
- Do not add unnecessary labels.
- Do not remove labels unnecessarily.
- Return only valid JSON.
- Do not include explanations or Markdown.

Required output format:
{{
    "labels_to_add": [],
    "labels_to_remove": [],
    "is_action_required": false
}}
"""

Conversation dataset

In [9]:
def create_conversation(row):

    input_data = {
        "from": str(row["from"]),
        "subject": str(row["subject"]),
        "body": str(row["body"]),
        "sender_type": str(row["sender_type"])
    }

    output_data = {
        "labels_to_add": parse_labels(row["labels_to_add"]),
        "labels_to_remove": parse_labels(row["labels_to_remove"]),
        "is_action_required": bool(row["is_action_required"])
    }

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": json.dumps(input_data, ensure_ascii=False)
        },
        {
            "role": "assistant",
            "content": json.dumps(output_data, ensure_ascii=False)
        }
    ]

    return {"messages": messages}


conversations = [
    create_conversation(row)
    for _, row in df.iterrows()
]

print("Total conversations:", len(conversations))

Total conversations: 1981


In [10]:
with open("email_conversations.jsonl", "w", encoding="utf-8") as f:

    for example in conversations:
        f.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Dataset saved successfully!")

Dataset saved successfully!


Split the dataset

In [11]:
train_data, temp_data = train_test_split(
    conversations,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Training: 1584
Validation: 198
Test: 199


In [12]:
def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for record in data:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

save_jsonl(train_data, "train.jsonl")
save_jsonl(validation_data, "validation.jsonl")
save_jsonl(test_data, "test.jsonl")

print("All datasets saved successfully!")

All datasets saved successfully!


Check GPU Availability Status

In [13]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU detected. Please enable GPU in Colab.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


Set Model configurations

In [14]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Load the base model

In [16]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Configure LoRA config for Fine-tuning

In [17]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.config.use_cache = False

In [18]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

Attach LoRA Adapters to the base model

In [19]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Load training and validation datasets

In [20]:
data_files = {
    "train": "train.jsonl",
    "validation": "validation.jsonl"
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1584
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 198
    })
})


Format Dataset

In [21]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

formatted_dataset = dataset.map(format_chat)

print(formatted_dataset["train"][0]["text"])

Map:   0%|          | 0/1584 [00:00<?, ? examples/s]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

<|im_start|>system

You are an AI email management assistant.

Analyze the provided email and determine:

1. Which labels should be added.
2. Which existing labels should be removed.
3. Whether the email requires an action or reply from the user.

Allowed labels:
["Adrenalin", "Arise/Website", "Azure/Invoice", "Azure/Promotions", "Azure/Support", "Azure/Updates", "Community/updates", "Company/Notice", "Company/Notices", "CraftSilicon", "Devops/BuildFailed", "Devops/BuildSuccess", "Devops/Updates", "Github/Updates", "Google Play Console", "Google Play console", "Google/Updates", "INBOX", "Lenovo/RepairService", "Lenovo/Service", "Lenovo/Services", "MSG91", "MongoDb/Atlas", "Newsletter", "Newsletters", "Playstore/Updates", "Projects", "Projects/AMS", "Projects/AR ConveyPay", "Projects/AR Field Connect", "Projects/ARConveyPay", "Projects/AROnboard", "Projects/Adernalin", "Projects/Arise Website", "Projects/AriseSakthi", "Projects/BRConnect", "Projects/ForgetPassword", "Projects/Forgetpass

Configure training parameters

In [22]:
training_args = SFTConfig(

    output_dir="./qwen-email-finetuned",

    # Training
    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    # Memory optimization
    fp16=True,
    bf16=False,

    gradient_checkpointing=True,

    # Evaluation
    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=25,

    logging_steps=5,

    # Early stopping support
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Dataset
    max_length=2048,
    packing=False,

    # Other
    report_to="none",

    save_total_limit=2,

    seed=42
)

 Initialize SFTTrainer

In [23]:
trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=formatted_dataset["train"],

    eval_dataset=formatted_dataset["validation"],

    processing_class=tokenizer,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3
        )
    ]
)

Tokenizing train dataset:   0%|          | 0/1584 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1584 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1584 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1584 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Check training configuration

In [24]:
print("Trainer initialized successfully!")

print("Training examples:",
      len(formatted_dataset["train"]))

print("Validation examples:",
      len(formatted_dataset["validation"]))

print("Trainable parameters:")

model.print_trainable_parameters()

Trainer initialized successfully!
Training examples: 1584
Validation examples: 198
Trainable parameters:
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
